# Standalone code-aware neighbourhood membership inference

Our research attack: mean neighbour normalized causal token NLL minus original NLL. No LUMIA probes or hidden states are loaded. Use the exact saved SERSEM config and heldout manifest. All perturbations are MLM proposals with conservative syntax/AST checks; failures are recorded, not substituted with the original or a zero score. This notebook is statically checked only; run manually on Kaggle GPU.

Start with the saved SERSEM config, existing **1,000-sample** held-out manifest and completed LUMIA/grey-box/fusion scores. Do not rerun training or create a new manifest. A smoke run explicitly evaluates a balanced subset of those IDs. Full mode evaluates every ID in the saved manifest. K defaults to 5. Site selection reuses the exact structural/token weights from the existing SERSEM reconstruction (not recovered author weights). Select `uniform` for the previous selection policy. No local-window NLL or neighbourhood/LUMIA fusion is implemented.


In [ ]:
from pathlib import Path
SOURCE_MODE = "uploaded"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
UPLOAD_SOURCE = INPUT_ROOT / "datasets/sayonsujitmondal/sersem-kaggle-source3"
UPLOAD_ZIP = UPLOAD_SOURCE / "neighbourhood_kaggle_source.zip"  # edit Dataset slug/mount
PROJECT_REPO_URL = ""
PROJECT_REF = ""
PROJECT_ROOT = WORK_ROOT / "sersem-neighbourhood-importance-v2"
UPSTREAM_NAME = "SERSEM-Poisoned-Chalice-Competition-2026"
UPSTREAM_SHA = "413f56040e5b4805bcf15ed794dec56bc4e16b41"
UPSTREAM_URL = "https://github.com/Serdark4ra/SERSEM-Poisoned-Chalice-Competition-2026.git"
# Same-session working outputs or a trusted saved run mounted under /kaggle/input.
BASELINE_RUN = WORK_ROOT / "sersem_outputs/starcoder2-3b-resource-validation-v1"
BASELINE_SCORES = BASELINE_RUN / "heldout_test_scores.parquet"
MODE = "smoke"  # smoke | full; full means all IDs in baseline manifest, not resampling
SMOKE_PER_LANGUAGE_CLASS = 3  # 30 balanced examples across 5 languages
RUN_ID = "neighbourhood-sersem-k5-smoke-v2"
OUTPUT_ROOT = WORK_ROOT / "neighbourhood_outputs"
RESUME_FROM = ""  # trusted saved neighbourhood run directory; no baseline feature copy
K = 5  # requested experiment; 1 | 3 retained for separate runs
SITE_IMPORTANCE = "sersem"  # sersem | uniform (previous uniform-within-kind policy)
NLL_REUSE_FROM = ""  # previous compatible neighbourhood run to reuse exact original/neighbour NLLs across K
TARGET_CACHE_DIR = ""  # empty reuses saved SERSEM model-cache path; set mounted cache if relocated
TARGET_LOCAL_FILES_ONLY = True
SOURCE_ACCESS = "cached"  # cached Arrow or explicit streaming; same pinned dataset
SOURCE_CHUNK_SIZE = 32
TARGET_GPU_BATCH_SIZE = 1  # 8192 cap; raise only after measuring memory; never auto-reduced
NLL_LOGIT_CHUNK_SIZE = 64
MLM_ID = "microsoft/codebert-base-mlm"
MLM_REVISION = "main"  # resolved and persisted once; no model download occurs locally
MLM_CACHE_DIR = WORK_ROOT / "hf_cache/hub"
MLM_LOCAL_FILES_ONLY = False  # explicit Kaggle download permission for this ~0.5 GB checkpoint
MLM_GPU_BATCH_SIZE = 4
MLM_MAX_LENGTH = 512  # MLM proposal window only, NEVER the StarCoder2 scoring context
MLM_TOP_K = 40
MAX_SITES = 12
ALLOW_SAFE_PYTHON_LOCAL_RENAME = True
ALLOW_COMMENT_INSERTION = True  # plain AST-verified comment when substitution is unavailable
PARK_MLM_ON_CPU = True  # GPU inference only; park weights in host RAM during target forwards
TEXT_CACHE_GIB = 0.10
SESSION_BUDGET_MINUTES = 480
WORK_ROOT.mkdir(parents=True,exist_ok=True)

# Explicit fresh-session preparation (runs only on Kaggle, not during source checks).
AUTO_DOWNLOAD_MODEL_IF_MISSING = True
DOWNLOAD_MIN_FREE_GIB = 3.0
AUTO_PREPARE_DATASET_IF_MISSING = True
DATASET_PREPARATION_BUDGET_GIB = 8.0  # conservative peak allowance when metadata is incomplete
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"  # edit for an attached Arrow cache


## Source and dependency setup
Attach the rebuilt source ZIP, saved SERSEM run, target cache and dataset cache. Source verification/install below use the same helpers as notebook 01. Set HF_DATASETS_CACHE before executing these cells if your Arrow cache is mounted elsewhere. Internet is needed only for explicitly allowed downloads/metadata; no submission contract is being targeted.


In [ ]:
import subprocess, shutil, zipfile, hashlib, json, sys, os
EXPECTED_FRAMEWORK_SHA256 = {'sersem_baseline.py': '9263371c56a5a1087f8a4e8d2adc9371f4461c1ffbf051c6fcee459c269e8a57', 'weighted_anomaly.py': '41aa9b87aa3dcfd86118c3618732f48a9a21e111d97e7e6322a41e8eef54573e', 'kaggle_setup.py': '990ad092df298db20ab9203d00b8bd5f108ab521154356405475f9595843cfb6', 'resource_data.py': 'ff737f86fa9084d5c72a9be45b5b24a54ac67748acbe45fcd89d7e9b96fd82d6', 'resource_pipeline.py': '5565efc812ffef8cd054b7dedc022dda0bdb1c493cd3a63815f3a5add3fbb4c8', 'research_execution.py': '348152e2f401cd0eaf997c604b8dba200eca43e3409159c2d6cc8bef7722e477', 'cache_setup.py': '560d6f2dd77530d2f461c0ad247f1fb7c2200336e6a37d99b25acde80bc7349a', 'requirements-kaggle.txt': '2dd3ad3196b4cdbba4d223df6b731e4f95b3ac468a216d84ce72eb3ead5e2d37', 'upstream_manifest.json': 'f8cb8621f0e29f1ac2864aa805449f21a59f528a9bcf12f34012a1f75ea0fbc1'}
# Prefer Kaggle's already-extracted source; never re-zip or copy datasets/weights.
if not (PROJECT_ROOT / "shared/sersem_baseline.py").exists():
    if PROJECT_ROOT.exists():
        raise FileExistsError(f"Existing incomplete project at {PROJECT_ROOT}; select a fresh PROJECT_ROOT")
    if SOURCE_MODE == "github":
        if not PROJECT_REPO_URL or not PROJECT_REF:
            raise ValueError("Set framework repository URL and immutable commit")
        subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", PROJECT_REF], check=True)
    elif SOURCE_MODE == "uploaded":
        if (UPLOAD_SOURCE / "shared/sersem_baseline.py").is_file():
            source_files = ["shared/" + name for name in EXPECTED_FRAMEWORK_SHA256]
            source_files += ["shared/author_source_search.json", "neighbourhood_attack/runner.py", "neighbourhood_attack/perturb.py", "neighbourhood_attack/README.md"]
            source_files += [UPSTREAM_NAME + "/" + name for name in ["README.md", "requirements.txt", "run_pipeline.sh", "code/ASTExtractor.py", "code/LinterExtractor.py", "code/LumiaAttack.py", "code/MIAttack.py", "code/plot_results.py", "code/run.py", "code/transformers_compat.py"]]
            for name in source_files:
                src = UPLOAD_SOURCE / name
                if not src.is_file():
                    raise FileNotFoundError(f"Extracted source is incomplete/outdated: {src}. Upload the rebuilt research source package.")
            PROJECT_ROOT.mkdir(parents=True)
            for name in source_files:
                dest = PROJECT_ROOT / name
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(UPLOAD_SOURCE / name,dest)
            for name in ["README.md","AUDIT.md","KAGGLE_RESOURCES.md","RESEARCH_RUN.md"]:
                if (UPLOAD_SOURCE / name).is_file():
                    shutil.copy2(UPLOAD_SOURCE / name,PROJECT_ROOT / name)
            print("Copied allowlisted extracted source:",UPLOAD_SOURCE)
        elif UPLOAD_ZIP.is_file():
            with zipfile.ZipFile(UPLOAD_ZIP) as archive:
                for item in archive.infolist():
                    dest=(PROJECT_ROOT / item.filename).resolve()
                    if not dest.is_relative_to(PROJECT_ROOT.resolve()) or any(part in {".env",".git"} for part in Path(item.filename).parts):
                        raise ValueError("Unsafe archive path")
                archive.extractall(PROJECT_ROOT)
            print("Extracted source ZIP:",UPLOAD_ZIP)
        else:
            raise FileNotFoundError(f"Neither extracted source at {UPLOAD_SOURCE} nor ZIP at {UPLOAD_ZIP}. Check your Kaggle Dataset mount.")
    else:
        raise ValueError("Unknown SOURCE_MODE")
# Bind the upstream provenance manifest itself before reading its contents.
for name, expected_hash in EXPECTED_FRAMEWORK_SHA256.items():
    helper_path=PROJECT_ROOT / "shared" / name
    if not helper_path.is_file() or hashlib.sha256(helper_path.read_bytes()).hexdigest()!=expected_hash:
        raise RuntimeError(f"Outdated framework file: {name}. Update the source Dataset and choose a fresh PROJECT_ROOT; existing source was not overwritten.")
UPSTREAM=PROJECT_ROOT / UPSTREAM_NAME
if not (UPSTREAM / "code/LumiaAttack.py").exists():
    if SOURCE_MODE != "github":
        raise FileNotFoundError("Uploaded package lacks the pinned upstream sources")
    subprocess.run(["git","clone",UPSTREAM_URL,str(UPSTREAM)],check=True)
    subprocess.run(["git","-C",str(UPSTREAM),"checkout","--detach",UPSTREAM_SHA],check=True)
manifest=json.loads((PROJECT_ROOT / "shared/upstream_manifest.json").read_text())
if manifest["commit_sha"]!=UPSTREAM_SHA:
    raise RuntimeError("Unexpected upstream provenance commit")
if (UPSTREAM / ".git").exists():
    actual=subprocess.check_output(["git","-C",str(UPSTREAM),"rev-parse","HEAD"],text=True).strip()
    if actual!=UPSTREAM_SHA:
        raise RuntimeError("Unexpected upstream HEAD")
for name, expected in manifest["source_sha256"].items():
    if hashlib.sha256((UPSTREAM / name).read_bytes()).hexdigest()!=expected:
        raise RuntimeError(f"Upstream source changed: {name}")
print("Verified pinned upstream source:",UPSTREAM_SHA)
# Verify standalone neighbourhood source as well.
EXPECTED_NEIGHBOURHOOD_SHA256 = {'neighbourhood_attack/runner.py': '7c62ed01c13faaeeb4fa52e78252cb18a98bae197921c9e32391244a1c409cc2', 'neighbourhood_attack/perturb.py': '031d7b66011f0b9711639eb130dc2439badc525b97d8dea1799bb8d68c56f706'}
for relative_name, expected_hash in EXPECTED_NEIGHBOURHOOD_SHA256.items():
    if hashlib.sha256((PROJECT_ROOT / relative_name).read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError("Outdated neighbourhood helper; use refreshed source and a fresh PROJECT_ROOT")
# End neighbourhood source verification.
sys.path.insert(0, str(PROJECT_ROOT))
for module_name in ["shared.kaggle_setup","shared.sersem_baseline","shared.weighted_anomaly","shared.resource_data","shared.resource_pipeline","shared.research_execution","shared.cache_setup","neighbourhood_attack.runner","neighbourhood_attack.perturb"]:
    loaded_module=sys.modules.get(module_name)
    if loaded_module is not None and not Path(loaded_module.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
        raise RuntimeError("Helpers from another project are loaded; restart the kernel and run in order")


In [ ]:
# 3. Install once, keeping Kaggle's compatible numerical/CUDA stack intact.
import sys, os, subprocess
from pathlib import Path
from shared.kaggle_setup import install_dependencies
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"  # avoid optional transport chunk-cache duplication
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
requirements_file = PROJECT_ROOT / "shared/requirements-kaggle.txt"
# Check system packages without importing Enchant or any numerical library.
system_packages = ["libenchant-2-2", "hunspell-en-us"]
system_status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *system_packages],
                               capture_output=True, text=True)
if system_status.returncode != 0 or system_status.stdout.splitlines().count("install ok installed") != len(system_packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *system_packages], check=True)
installation_report = install_dependencies(requirements_file, WORK_ROOT)
print("Dependencies ready. Proceed to the initialization cell.")
# Optional authentication: uncomment if needed; never print/save the token.
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")


In [ ]:
import json, hashlib, random, sys
from pathlib import Path
from copy import deepcopy
from huggingface_hub import HfApi
from shared.kaggle_setup import validate_environment
from shared.sersem_baseline import write_json
from shared.resource_data import SourceStore
from shared.resource_pipeline import Monitor, restore_run
from neighbourhood_attack.runner import load_target, run as run_neighbourhood, validate_heldout_manifest, validate_saved_scores
import numpy as np
import pandas as pd
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU. CPU inference is not supported.")
environment_report = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt")
if K not in {1,3,5} or MODE not in {"smoke","full"}:
    raise ValueError("Choose K=1,3,5 and smoke/full mode")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".",".."}:
    raise ValueError("RUN_ID must be a directory name")
if not all(isinstance(x,int) and x>0 for x in [SOURCE_CHUNK_SIZE,TARGET_GPU_BATCH_SIZE,MLM_GPU_BATCH_SIZE,NLL_LOGIT_CHUNK_SIZE,MAX_SITES,SMOKE_PER_LANGUAGE_CLASS]):
    raise ValueError("Chunk/batch/site/sample counts must be positive integers")
if SOURCE_ACCESS not in {"cached","streaming"} or not 8 <= MLM_MAX_LENGTH <= 512 or not 1 <= MLM_TOP_K <= 100:
    raise ValueError("Invalid source or MLM proposal configuration")
base_cfg = json.loads((BASELINE_RUN / "config.json").read_text())
if base_cfg["versions"] != environment_report["versions"]:
    raise ValueError("SERSEM and neighbourhood API/numerical versions differ. Restore the baseline environment before comparing.")
if base_cfg["upstream_sha"] != UPSTREAM_SHA:
    raise ValueError("Unexpected SERSEM upstream provenance")
manifest = pd.read_parquet(BASELINE_RUN / "heldout_test_manifest.parquet")
validate_heldout_manifest(manifest)
full_manifest_sha256 = hashlib.sha256(manifest.to_json(orient="records").encode()).hexdigest()
if SITE_IMPORTANCE not in {"sersem", "uniform"}:
    raise ValueError("Choose sersem or uniform site importance")
if not BASELINE_SCORES.is_file():
    raise FileNotFoundError("Attach saved LUMIA, grey-box and fusion scores; no retraining is performed")
validate_saved_scores(manifest, pd.read_parquet(BASELINE_SCORES))
baseline_scores_sha256 = hashlib.sha256(BASELINE_SCORES.read_bytes()).hexdigest()
# CPU-only structural setup uses the SAME pinned upstream extractors as SERSEM.
sys.path.insert(0, str(UPSTREAM / "code"))
if SITE_IMPORTANCE == "sersem":
    from shared.weighted_anomaly import StructuralWeights
    StructuralWeights()  # fail before model loading if exact dependencies are missing
if MODE == "smoke":
    manifest = manifest.groupby(["language","is_member"],group_keys=False).head(SMOKE_PER_LANGUAGE_CLASS).copy()
manifest = manifest.reset_index(drop=True)
OUT = OUTPUT_ROOT / RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM,OUT)
saved_path = OUT / "config.json"
saved_cfg = json.loads(saved_path.read_text()) if saved_path.exists() else None
mlm_revision = saved_cfg["mlm_revision"] if saved_cfg and saved_cfg["mlm_id"]==MLM_ID and saved_cfg["requested_mlm_revision"]==MLM_REVISION else HfApi().model_info(MLM_ID,revision=MLM_REVISION).sha
if not mlm_revision:
    raise ValueError("MLM revision did not resolve to an immutable SHA")
random.seed(base_cfg["seed"]); np.random.seed(base_cfg["seed"]); torch.manual_seed(base_cfg["seed"]); torch.cuda.manual_seed_all(base_cfg["seed"])
cfg = dict(model_id=base_cfg["model_id"],model_revision=base_cfg["model_revision"],dtype=base_cfg["dtype"],
           max_length=base_cfg["max_length"],seed=base_cfg["seed"],dataset_id=base_cfg["dataset_id"],dataset_revision=base_cfg["dataset_revision"],
           target_nll_version="causal-next-token-float32-ce-float64-sum-v1",k=K,mode=MODE,site_importance=SITE_IMPORTANCE,
           structural_weight_provenance="existing SERSEM Section 2.1 reconstruction; max target-token weight over affected spans; insertion=0",
           full_heldout_manifest_sha256=full_manifest_sha256,baseline_scores_sha256=baseline_scores_sha256,nll_reuse_from=NLL_REUSE_FROM,
           heldout_manifest_sha256=hashlib.sha256(manifest.to_json(orient="records").encode()).hexdigest(),
           model_cache_dir=TARGET_CACHE_DIR or base_cfg["resources"]["model_cache_dir"],model_local_files_only=TARGET_LOCAL_FILES_ONLY,
           mlm_id=MLM_ID,mlm_revision=mlm_revision,requested_mlm_revision=MLM_REVISION,mlm_cache_dir=str(MLM_CACHE_DIR),mlm_local_files_only=MLM_LOCAL_FILES_ONLY,
           mlm_batch_size=MLM_GPU_BATCH_SIZE,mlm_max_length=MLM_MAX_LENGTH,mlm_top_k=MLM_TOP_K,max_sites=MAX_SITES,
           allow_python_local_rename=ALLOW_SAFE_PYTHON_LOCAL_RENAME,allow_comment_insertion=ALLOW_COMMENT_INSERTION,park_mlm_on_cpu=PARK_MLM_ON_CPU,
           target_batch_size=TARGET_GPU_BATCH_SIZE,nll_logit_chunk_size=NLL_LOGIT_CHUNK_SIZE,source_chunk_size=SOURCE_CHUNK_SIZE,
           versions=environment_report["versions"],baseline_runner_sha256=base_cfg["runner_sha256"],
           runner_sha256=hashlib.sha256(b"".join((PROJECT_ROOT / name).read_bytes() for name in ["neighbourhood_attack/runner.py","neighbourhood_attack/perturb.py","shared/research_execution.py","shared/weighted_anomaly.py"])).hexdigest())
# Operational cache paths/batch sizes may change on resume; model/math/cohort cannot.
operational = {"model_cache_dir","mlm_cache_dir","model_local_files_only","mlm_local_files_only","target_batch_size","mlm_batch_size","source_chunk_size","nll_reuse_from"}
if saved_cfg and {k:v for k,v in saved_cfg.items() if k not in operational} != {k:v for k,v in cfg.items() if k not in operational}:
    raise ValueError("Neighbourhood configuration/cohort/version changed; use a NEW RUN_ID")
write_json(OUT / "config.json",cfg)
write_json(OUT / "environment.json",environment_report)
manifest.to_parquet(OUT / "heldout_manifest.parquet",index=False)
source_cfg = deepcopy(base_cfg)
source_cfg["resources"]["data_access"] = SOURCE_ACCESS
source_cfg["resources"]["text_cache_gib"] = TEXT_CACHE_GIB
source_cfg["resources"]["session_budget_minutes"] = SESSION_BUDGET_MINUTES
monitor = Monitor(source_cfg,OUT)
monitor.tick("neighbourhood_initialized",0,len(manifest),force=True)
monitor.guard()
source_store = SourceStore(source_cfg,OUT,monitor)
print("Pinned target:",cfg["model_id"],cfg["model_revision"],"context",cfg["max_length"],"dtype",cfg["dtype"])
print("Requested rows:",len(manifest),manifest.groupby(["language","is_member"]).size().to_dict())


## Explicit cache preparation
Reuse a complete pinned snapshot, otherwise download only required model/tokenizer files when explicitly enabled. Cached data mode may prepare missing Arrow files; streaming is a separate explicit choice. At least 3 GiB is reserved for working files after setup; metadata/budget checks can fail before download. These are setup operations, not inference.


In [ ]:
from shared.cache_setup import ensure_model_snapshot, ensure_dataset_cache
if DOWNLOAD_MIN_FREE_GIB < 3:
    raise ValueError("Keep at least 3 GiB free after asset preparation")
# Always use the SHA inherited from the SERSEM run; never resolve a new target main.
model_cache_report = ensure_model_snapshot(cfg["model_id"],cfg["model_revision"],cfg["model_cache_dir"],
                     AUTO_DOWNLOAD_MODEL_IF_MISSING,DOWNLOAD_MIN_FREE_GIB,WORK_ROOT,OUT / "model_cache_setup.json")
dataset_cache_report = ensure_dataset_cache(source_cfg,AUTO_PREPARE_DATASET_IF_MISSING,
                       DATASET_PREPARATION_BUDGET_GIB,DOWNLOAD_MIN_FREE_GIB,WORK_ROOT,OUT / "dataset_cache_setup.json")
print("Same pinned target/source cache ready; no LUMIA probes or hidden states loaded.")


## Run independently and resume

The original NLL, generated texts/failures, neighbour NLLs and completed scalar scores commit to compressed SQLite with checksums. CodeBERT's MLM head is checked for missing/randomly initialized tensors. The MLM runs on GPU; by default its weights are parked in host RAM before StarCoder2 scoring. StarCoder2 never loads hidden states or LUMIA probes. Start with batch=1 at the inherited context; an OOM raises instead of silently truncating or rounding.

Failures remain explicit rows with no fabricated score. A complete comparison needs successful neighbours for every requested row. If not, reports state coverage and recompute **both attacks on exactly the same successful intersection**, verifying labels/language/content hashes. No full-cohort result is claimed for that subset. Neighbours and all failed proposal reasons remain in the SQLite generation records. No K reduction or original-as-neighbour fallback occurs.


In [ ]:
# Reuse a live target only when its model/revision/precision/context agree.
model = globals().get("model")
tokenizer = globals().get("tokenizer")
target_identity = {k:cfg[k] for k in ["model_id","model_revision","dtype","max_length"]}
if model is None:
    model, tokenizer = load_target(cfg)
    model._neighbourhood_identity = target_identity
elif getattr(model,"_neighbourhood_identity",None) != target_identity or tokenizer is None:
    raise RuntimeError("A different target is already loaded; restart the kernel.")
report = run_neighbourhood(cfg,manifest,source_store,model,tokenizer,OUT,monitor,str(BASELINE_SCORES))
print(json.dumps(report,indent=2))


In [ ]:
# Rerun this reporting-only cell later if SERSEM scores became available.
from neighbourhood_attack.runner import evaluate_results
result_path = OUT / "neighbourhood_scores.parquet"
if result_path.exists():
    results = pd.read_parquet(result_path)
    comparison = evaluate_results(results,manifest,str(BASELINE_SCORES),OUT)
    print(json.dumps(comparison,indent=2))
else:
    print("Run interrupted before final export; restore the saved run and rerun scoring to reuse caches.")


## Save before ending a Kaggle session

Persist the **entire** OUT directory with Save Version **with outputs**, a private checkpoint Dataset or external copy. Wait for writes to finish. Include config/environment, heldout manifest, neighbourhood.sqlite (generation texts, per-original/per-neighbour NLLs, results/checksums), any accompanying SQLite journal, metrics/runtime_history and milestone Parquets at 500 completed examples (and larger legacy milestones when applicable). Compressed cached neighbour texts still grow with file size and K: monitor disk, monitor the K=5 checkpoint size. No full duplicate ZIP of checkpoints is created.

Next session attach those outputs, the original SERSEM manifest/config, source and target/MLM caches separately. Set RESUME_FROM to the trusted saved OUT, same RUN_ID/K/math/cohort/versions, and new mounted cache paths. Completed originals are not re-evaluated. Source files must still be accessible for remaining rows. Neither source nor pretrained target/MLM weights are copied into the result directory.

runtime.json and runtime_history.jsonl use **measured** generation+scoring+I/O elapsed time and observed examples/second; remaining ETA is tentative while length/cache mix changes. No runtime promise is made. Smoke/full/K changes require a new run ID; exact target NLLs can be reused in the same run on restart. Final ROC/overlap thresholds use heldout labels for reporting only, never perturbation/score fitting. No neighbourhood fusion, local-window NLL, or submission restructuring is included.


## Smoke test, then the full 1,000-example run

1. Attach the rebuilt `neighbourhood_kaggle_source.zip`, the existing SERSEM run (including its exact 1,000-row manifest and all three saved score columns), and pinned model/dataset caches. Edit `UPLOAD_SOURCE`, `BASELINE_RUN` and cache paths. Keep the inherited StarCoder2 revision/context/precision. No LUMIA or StarCoder2 training occurs.
2. Smoke: `MODE="smoke"`, `K=5`, `SITE_IMPORTANCE="sersem"`, `SMOKE_PER_LANGUAGE_CLASS=3`, `RUN_ID="neighbourhood-sersem-k5-smoke-v2"`. Run cells in order on Kaggle GPU. This selects up to 30 existing IDs without resampling. Inspect coverage, failure reasons, selected-site weights/locations and neighbour transformations in the score export/SQLite. Each successful example must have exactly five distinct neighbours. TPR at 1% FPR is resolution-limited on this small smoke cohort.
3. Full: restart the kernel, set `MODE="full"`, `RUN_ID="neighbourhood-sersem-k5-full-v2"`, retain `K=5` and `SITE_IMPORTANCE="sersem"`. Leave `RESUME_FROM=""` for the first full session. The saved manifest must contain exactly 1,000 rows; all are requested. Optional `NLL_REUSE_FROM` can point to the compatible smoke output (strict identity checks apply). Report `metrics.json`: ROC-AUC, empirical TPR at FPR <=1%, coverage by language/label, per-language metrics, and `matched_metrics` recomputed for all four score columns on the common successful cohort.
4. Persist the entire output directory. To resume, retain the full run ID/config and set `RESUME_FROM` to its saved directory. For a uniform ablation, use `SITE_IMPORTANCE="uniform"` and a new run ID; this preserves the previous uniform site-selection policy while retaining the diversity-first generation pass. Importance mode/code/cohort changes invalidate generation checkpoints.
